# [프로젝트] 문서 요약 시스템 만들기

LLM을 사용하여, PDF 문서의 요약을 생성하는 어플리케이션을 구현해 보겠습니다.

1. Arxiv API에 검색어를 입력하여, 논문 정보를 가져옵니다.
2. Abstract와 Full Text를 불러옵니다. Full Text는 작은 크기로 청킹(Chunking)합니다.   
3. Step Back Prompting을 이용하여, Abstract의 내용을 바탕으로 요약의 포인트를 생성합니다.
4. 요약 포인트를 이용하여 청크별 요약을 생성합니다.
5. 최종 요약본을 생성합니다.

In [ ]:
#!pip install openai arxiv langchain==0.3.27 langchain_community==0.3.27 langchain-openai langchain-google-genai langchain-community tiktoken pymupdf

In [4]:
!pip install openai langchain-openai langchain-google-genai tiktoken pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 50.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [5]:
!pip uninstall -y arxiv langchain-community langchain

!pip install -U \
    langchain \
    langchain-community \
    arxiv

Found existing installation: langchain 1.3.13
Uninstalling langchain-1.3.13:
  Successfully uninstalled langchain-1.3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
##!pip uninstall -y arxiv
##!pip install "arxiv<2.0"

Found existing installation: arxiv 1.4.8
Uninstalling arxiv-1.4.8:
  Successfully uninstalled arxiv-1.4.8
  Using cached arxiv-1.4.8-py3-none-any.whl.metadata (8.1 kB)
Using cached arxiv-1.4.8-py3-none-any.whl (12 kB)


## 기본 LLM 불러오기

In [6]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

In [7]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
try:
    #llm = ChatOpenAI(model='gpt-5-mini', temperature=1.0)
    llm = ChatOpenAI(model='gpt-5.5')
    print("✅ GPT API 사용 가능!")
except:
    print("❌ GPT API 사용 불가- API 키를 확인하세요!")
try:
    llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7,
                                rate_limiter=rate_limiter)
    print("✅ Gemini API 사용 가능!")
except:
    print("❌ Gemini API 사용 불가- API 키를 확인하세요!")

✅ GPT API 사용 가능!
✅ Gemini API 사용 가능!


필수 라이브러리를 불러옵니다.

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("필수 모듈 임포트 완료")

필수 모듈 임포트 완료


# Preliminary: LangChain의 Document Loader와 Text Splitter

LangChain은 외부 데이터와의 연결을 위한 다양한 도구를 제공합니다.

### 1) 랭체인 Document Loader

Document Loader는 파일 경로를 입력받아 데이터를 불러옵니다.

`langchain_community.document_loaders` (https://python.langchain.com/docs/integrations/document_loaders/)    

- CSVLoader, PyMuPDFLoader, WebBaseLoader 등의 다양한 로더가 존재합니다.

In [9]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader, CSVLoader
from langchain_community.document_loaders import ArxivLoader

/tmp/ipykernel_643/2129085803.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader, CSVLoader



로더는 load() 또는 비동기 방식으로 실행할 수 있으며, `Document` 클래스로 저장됩니다.
- metadata: Dict 형식으로 값이 저장됩니다.
- page_content: string 형태의 실제 문서 내용이 저장됩니다.


Data Loader와 Text Splitter   
이번 실습에서는 ArxivLoader를 사용합니다.

In [10]:
from rich import print as rprint

In [ ]:
## Jump!!
# import arxiv

# def load_arxiv_docs(query, max_chars=100000):


#   client = arxiv.Client(
#     delay_seconds=5,
#     num_retries=5)

#   search = arxiv.Search(
#     query=query,
#     max_results=3)

#   #return next(client.results(search))
#   return list(client.results(search))

# result = load_arxiv_docs("Korean Large Language Model Benchmark")
# #result = load_arxiv_docs("kanana Model Benchmark")[0]
# rprint(result[1])

# print(result[1].title)
# print(result[1].summary)
# print(result[1].entry_id)

http://arxiv.org/abs/2512.08480v1

Soft Inductive Bias Approach via Explicit Reasoning Perspectives in Inappropriate Utterance Detection Using Large Language Models
Recent incidents in certain online games and communities, where anonymity is guaranteed, show that unchecked inappropriate remarks frequently escalate into verbal abuse and even criminal behavior, raising significant social concerns. Consequently, there is a growing need for research on techniques that can detect inappropriate utterances within conversational texts to help build a safer communication environment. Although large-scale language models trained on Korean corpora and chain-of-thought reasoning have recently gained attention, research applying these approaches to inappropriate utterance detection remains limited. In this study, we propose a soft inductive bias approach that explicitly defines reasoning perspectives to guide the inference process, thereby promoting rational decision-making and preventing errors that may arise during reasoning. We f

In [11]:
from typing import List
import tempfile
import requests
import arxiv

from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader


class MyArxivLoader:

    def __init__(
        self,
        query: str,
        load_max_docs: int = 1,
        doc_content_chars_max: int = 100000,
    ):
        self.query = query
        self.load_max_docs = load_max_docs
        self.doc_content_chars_max = doc_content_chars_max

        self.client = arxiv.Client(
            page_size=load_max_docs,
            delay_seconds=3,
            num_retries=5,
        )

    def load(self) -> List[Document]:

        search = arxiv.Search(
            query=self.query,
            max_results=self.load_max_docs,
            sort_by=arxiv.SortCriterion.Relevance,
        )

        docs = []

        for paper in self.client.results(search):

            # -------------------------------
            # PDF 다운로드
            # -------------------------------
            response = requests.get(paper.pdf_url, timeout=60)
            response.raise_for_status()

            with tempfile.NamedTemporaryFile(
                suffix=".pdf",
                delete=False
            ) as f:

                f.write(response.content)
                pdf_path = f.name

            # -------------------------------
            # PDF 읽기
            # -------------------------------
            loader = PyMuPDFLoader(pdf_path)
            pages = loader.load()

            page_content = "\n".join(
                page.page_content for page in pages
            )

            page_content = page_content[: self.doc_content_chars_max]

            # -------------------------------
            # Metadata 생성
            # -------------------------------
            metadata = {
                "Title": paper.title,
                "Authors": [a.name for a in paper.authors],
                "Published": str(paper.published),
                "Updated": str(paper.updated),
                "Summary": paper.summary,
                "Entry ID": paper.entry_id,
                "PDF URL": paper.pdf_url,
                "Primary Category": paper.primary_category,
                "Categories": paper.categories,
                "Comment": paper.comment,
                "Journal Ref": paper.journal_ref,
                "DOI": paper.doi,
            }

            docs.append(
                Document(
                    page_content=page_content,
                    metadata=metadata,
                )
            )

        return docs

In [12]:
def load_arxiv_docs(query, max_chars=100000):
    loader = MyArxivLoader(
        query=query, # 검색어
        load_max_docs=1, # 최대 문서 수 (여기서는 1개만 사용)
        doc_content_chars_max=max_chars # 최대 문서 길이
        #load_all_available_meta=True, # 모든 메타데이터 로드
    )

    return loader.load()


result = load_arxiv_docs("Korean Large Language Model Benchmark")[0]
#result = load_arxiv_docs("kanana Model Benchmark")[0]
rprint(result)

Document(
    metadata={
        'Title': 'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context',
        'Authors': ['Xiaonan Wang', 'Jinyoung Yeo', 'Joon-Ho Lim', 'Hansaem Kim'],
        'Published': '2024-12-10 07:20:51+00:00',
        'Updated': '2024-12-10 07:20:51+00:00',
        'Summary': "Large language models have exhibited significant enhancements in performance across various 
tasks. However, the complexity of their evaluation increases as these models generate more fluent and coherent 
content. Current multilingual benchmarks often use translated English versions, which may incorporate Western 
cultural biases that do not accurately assess other languages and cultures. To address this research gap, we 
introduce KULTURE Bench, an evaluation framework specifically designed for Korean culture that features datasets of
cultural news, idioms, and poetry. It is designed to assess language models' cultural comprehension and reasoning 
capabilities at the word, sentence, and paragraph levels. Using the KULTURE Bench, we assessed the capabilities of 
models trained with different language corpora and analyzed the results comprehensively. The results show that 
there is still significant room for improvement in the models' understanding of texts related to the deeper aspects
of Korean culture.",
        'Entry ID': 'http://arxiv.org/abs/2412.07251v1',
        'PDF URL': 'https://arxiv.org/pdf/2412.07251v1',
        'Primary Category': 'cs.CL',
        'Categories': ['cs.CL'],
        'Comment': 'Accepted by the 38th Pacific Asia Conference on Language, Information and Computation',
        'Journal Ref': None,
        'DOI': None
    },
    page_content='Abstract \nLarge language models (LLMs) have \nexhibited significant enhancements in 
\nperformance across various tasks. However, \nthe complexity of their evaluation increases \nas these models 
generate more fluent and \ncoherent content. Current multilingual \nbenchmarks often use translated English 
\nversions, which may incorporate Western \ncultural biases that do not accurately assess \nother languages and 
cultures. To address \nthis research gap, we introduce KULTURE \nBench, \nan \nevaluation \nframework 
\nspecifically designed for Korean culture \nthat features datasets of cultural news, \nidioms, and poetry. It is 
designed to assess \nlanguage models\' cultural comprehension \nand reasoning capabilities at the word, \nsentence,
and paragraph levels. Using the \nKULTURE Bench, we assessed the \ncapabilities of models trained with different 
\nlanguage corpora and analyzed the results \ncomprehensively. The results show that \nthere \nis \nstill 
\nsignificant \nroom \nfor \nimprovement in the models’ understanding \nof texts related to the deeper aspects of 
\nKorean culture.  \n1 \nIntroduction \nThe proliferation of LLMs capable of generating \nfluent and plausible 
responses has significantly \ncomplicated the task of assessing their knowledge \nand reasoning abilities (Li et 
al., 2024). To address \nthis challenge, researchers have developed a range \nof benchmarks designed to assess 
diverse \ncapabilities of models such as GLUE (Wang et al., \n2018) and MMLU (Hendrycks et al., 2020). \nAlthough 
these benchmarks are primarily in \nEnglish, when addressing multilingual issues, \ncurrent evaluation practices 
often depend on \ntranslations of those datasets originally in English \n(Shi et al., 2022). However, a critical 
issue arises as \ntheir inherent Western cultural bias makes them \nunsuitable and inappropriate for evaluating 
LLMs \nacross a variety of cultures and languages. \nTherefore, there has been a notable trend toward \nthe 
development of culturally specific benchmarks \nthat cater to local contexts such as IndiBias (Sahoo \net al., 
2024) for Indian culture, Heron-Bench \n(Inoue et al., 2024) for Japanese culture, and \nCCPM (Li et al., 2021) and
ChID (Zheng et al, \n2019) for Chinese culture. However, compar

In [ ]:
## Jump!!

def load_arxiv_docs(query, max_chars=100000):
    loader = ArxivLoader(
        query=query, # 검색어
        load_max_docs=1, # 최대 문서 수 (여기서는 1개만 사용)
        doc_content_chars_max=max_chars, # 최대 문서 길이
        load_all_available_meta=True, # 모든 메타데이터 로드
    )
    return loader.load()


result = load_arxiv_docs("Korean Large Language Model Benchmark")[0].get("metadata")
#result = load_arxiv_docs("kanana Model Benchmark")[0]
rprint(result)

In [13]:
print('전체 글자 수:', len(result.page_content))
keys = [key for key in result.metadata.keys()]

전체 글자 수: 59724


In [14]:
print(keys)

['Title', 'Authors', 'Published', 'Updated', 'Summary', 'Entry ID', 'PDF URL', 'Primary Category', 'Categories', 'Comment', 'Journal Ref', 'DOI']


In [15]:
result

Document(metadata={'Title': 'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context', 'Authors': ['Xiaonan Wang', 'Jinyoung Yeo', 'Joon-Ho Lim', 'Hansaem Kim'], 'Published': '2024-12-10 07:20:51+00:00', 'Updated': '2024-12-10 07:20:51+00:00', 'Summary': "Large language models have exhibited significant enhancements in performance across various tasks. However, the complexity of their evaluation increases as these models generate more fluent and coherent content. Current multilingual benchmarks often use translated English versions, which may incorporate Western cultural biases that do not accurately assess other languages and cultures. To address this research gap, we introduce KULTURE Bench, an evaluation framework specifically designed for Korean culture that features datasets of cultural news, idioms, and poetry. It is designed to assess language models' cultural comprehension and reasoning capabilities at the word, sentence, and paragraph levels. Using t

### 2) 랭체인 Text Splitter

불러온 데이터는 랭체인의 `TextSplitter`를 통해 작은 단위로 분리합니다.

이를 청킹(Chunking)이라고 하는데, 이후의 RAG 파트에서 더 자세히 다룰 예정입니다.

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter: 문자나 토큰 단위로 분리하고, Overlap을 통해 내용 중첩
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000,
    chunk_overlap=2000,
) # 0- 10000, 8000- 18000, 16000 -26000
# 구분자(delimiter, separator 기준으로) 청킹 (줄바꿈 기호, 공백, 마침표, 쉼표)


chunks = text_splitter.split_documents([result])

print('Chunk 개수:', len(chunks), type(chunks))

Chunk 개수: 8 <class 'list'>


In [17]:
print(chunks[0].page_content)

Abstract 
Large language models (LLMs) have 
exhibited significant enhancements in 
performance across various tasks. However, 
the complexity of their evaluation increases 
as these models generate more fluent and 
coherent content. Current multilingual 
benchmarks often use translated English 
versions, which may incorporate Western 
cultural biases that do not accurately assess 
other languages and cultures. To address 
this research gap, we introduce KULTURE 
Bench, 
an 
evaluation 
framework 
specifically designed for Korean culture 
that features datasets of cultural news, 
idioms, and poetry. It is designed to assess 
language models' cultural comprehension 
and reasoning capabilities at the word, 
sentence, and paragraph levels. Using the 
KULTURE Bench, we assessed the 
capabilities of models trained with different 
language corpora and analyzed the results 
comprehensively. The results show that 
there 
is 
still 
significant 
room 
for 
improvement in the models’ understandi

이제, 텍스트 로드 결과를 활용하여 요약 시스템을 만들어 보세요!

In [ ]:
# 1. Stuff - 1번

# 2. Map-Reduce - Chunk 개수 + 1 - 병렬 처리가 가능하기 때문에 빠름
# 3. Refine - Chunk 개수 - 토큰 소모는 제일 큼

# 4. Hybrid -


## 2. Step-Back Prompting을 이용한 요약 가이드라인 생성 체인   

Step Back Prompting은 전체 답변을 작성하기 전 전체 구획이나 사전 정보를 준비하는 과정입니다.   
논문의 메타데이터를 입력하여 요약 가이드라인을 생성하는 체인을 구성하세요.    

-  LLM을 이용한 메타 프롬프팅을 추천합니다.

In [18]:
chunks[0].metadata

{'Title': 'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context',
 'Authors': ['Xiaonan Wang', 'Jinyoung Yeo', 'Joon-Ho Lim', 'Hansaem Kim'],
 'Published': '2024-12-10 07:20:51+00:00',
 'Updated': '2024-12-10 07:20:51+00:00',
 'Summary': "Large language models have exhibited significant enhancements in performance across various tasks. However, the complexity of their evaluation increases as these models generate more fluent and coherent content. Current multilingual benchmarks often use translated English versions, which may incorporate Western cultural biases that do not accurately assess other languages and cultures. To address this research gap, we introduce KULTURE Bench, an evaluation framework specifically designed for Korean culture that features datasets of cultural news, idioms, and poetry. It is designed to assess language models' cultural comprehension and reasoning capabilities at the word, sentence, and paragraph levels. Using the KULTURE Ben

In [19]:
from langchain_core.prompts import ChatPromptTemplate


guideline_prompt = ChatPromptTemplate(
    [
        ('system','''
당신은 학술 논문 분석 전문가입니다.

논문의 제목과 Abstract을 분석하여, 논문을 요약할 때 중점적으로 다루어야 하는 핵심 기준을 도출해 주세요.
핵심 가이드라인을 개조식으로 작성하고.가이드라인만 출력하세요.

아래의 문제를 중점적으로 고려하세요.

1) 핵심 연구 질문(무슨 문제에 답하려고 하는가?)
2) 방법론 (어떤 접근법이 중요한가)
3) 주요 기여도(학술적으로 어떤 가치가 있는가?)
4) 평가 방식 (결과의 의미와 한계)
5) 기술적 세부사항

최종 가이드라인은 2000자 이내로 작성하세요.
'''),
        ('human','''
논문의 제목: {Title}

논문의 Abstract: {Summary}
''')

    ]
)

guideliner = guideline_prompt | llm | StrOutputParser()

guide = guideliner.invoke(result.metadata)
# 매개변수에 해당하는 것만 전달

print(guide)

- **핵심 연구 질문**: 기존 다국어 벤치마크가 번역 기반·서구 문화 편향을 내포한다는 한계를 어떻게 극복하고, 언어모델의 **한국 문화 맥락 이해 및 추론 능력**을 더 정확히 평가할 수 있는가에 초점을 둔다.

- **문제 설정의 중요성**: 단순한 한국어 처리 능력이 아니라, 한국 문화에 내재된 의미·관습·표현·문학적 맥락을 모델이 얼마나 이해하는지를 평가하려는 연구임을 강조한다.

- **방법론**: KULTURE Bench가 한국 문화 평가에 특화된 벤치마크로 설계되었으며, **문화 뉴스, 관용어, 시** 데이터셋을 포함한다는 점을 중심적으로 설명한다.

- **평가 범위**: 벤치마크가 단어, 문장, 문단 수준에서 모델의 문화적 이해와 추론 능력을 평가하도록 구성되었다는 다층적 평가 구조를 다룬다.

- **비교 대상 모델**: 서로 다른 언어 말뭉치로 학습된 언어모델들을 평가했다는 점을 통해, 학습 데이터의 언어·문화적 구성과 성능 간 관계를 분석하는 데 주목한다.

- **주요 기여도**: 한국 문화 맥락에 특화된 평가 프레임워크를 제안함으로써, 기존 영어 중심·번역 중심 벤치마크의 한계를 보완하고 문화적 공정성과 지역 특화 평가의 필요성을 제기한 학술적 가치를 강조한다.

- **결과 해석**: 평가 결과, 현재 언어모델들이 한국 문화의 심층적 의미를 이해하는 데 여전히 부족함이 있음을 보여준다는 점을 핵심 결론으로 정리한다.

- **평가 방식의 의미**: 유창한 생성 능력만으로는 문화적 이해를 검증하기 어렵기 때문에, 문화 지식·맥락 추론·표현 이해를 분리해 평가하는 벤치마크의 필요성을 설명한다.

- **한계 및 고려점**: Abstract에 구체적 성능 수치, 모델 종류, 데이터 구축 방식, 평가 지표가 제시되지 않았으므로 요약 시 해당 세부사항은 본문 확인이 필요하다고 다룬다.

- **기술적 세부사항**: 데이터셋 구성, 문항 유형, 평가 레벨, 모델별 학습 코퍼스 차이, 정량·정성 분석 방식이 논문 요약에서 반드시 확인해야 할 

## 3. Map-Reduce 요약 프레임워크   

Map-Reduce 프레임워크는 Map-Reduce 과정으로 긴 문서를 요약하는 방식입니다.   
- Map: 전체 문서를 전체 문서를 부분으로 분리한 후, 각각 요약합니다.
- Reduce: 요약본을 모두 모아서, 최종 요약본을 작성합니다.

생성한 가이드라인을 이용해, 개별 문서를 요약하는 체인을 만들고 실행하세요.   

결과는 summaries에 리스트 형태로 저장하세요.

In [20]:
# Map 과정 : 각 문서에 대해 요약을 생성합니다.
from tqdm import tqdm
# 청크에 대한 요약 생성 (가이드라인 참고)

# 최종 요약만 한국어로 수행한다면 어떨까요?
map_prompt = ChatPromptTemplate(
    [
        ('system', '''주어진 논문의 내용을 요약하세요.
아래의 논문 일부와, 요약 가이드라인의 내용을 참고하여 상세한 요약문을 영어로 작성하세요.
요약문의 길이는 2-4개의 문단과 문단별 10문장 내외로 작성하세요.'''),
        ('human', '''
논문의 내용: {text}

---

요약 참고 가이드라인: {guide}''')
    ]
).partial(guide=guide)

map_chain  = map_prompt | llm | StrOutputParser()

summaries = await map_chain.abatch(chunks)
# 청크 개수만큼 LLM을 병렬 실행해서 요약문 생성
print(summaries)

/usr/local/lib/python3.12/dist-packages/langchain_core/prompts/chat.py:1215: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  kwargs = self._merge_partial_and_user_variables(**kwargs)


['The paper introduces **KULTURE Bench**, a benchmark designed to evaluate large language models’ understanding of **Korean cultural context**, rather than merely their ability to process Korean text. The central research problem is that many existing multilingual benchmarks are created by translating English datasets, which can preserve Western cultural assumptions and fail to measure knowledge, reasoning, and expression rooted in other cultures. As LLMs become increasingly fluent and coherent, surface-level generation quality is no longer sufficient for evaluating whether models truly understand culturally embedded meanings. The authors argue that Korean cultural comprehension requires assessing knowledge of idiomatic expressions, literary conventions, historical references, social context, and culturally specific forms of reasoning. Although several Korean-related benchmarks already exist, many rely mainly on question-answer formats and test general cultural knowledge or bias, rathe

In [21]:
summaries = map_chain.batch(chunks)

/usr/local/lib/python3.12/dist-packages/langchain_core/prompts/chat.py:1187: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  kwargs = self._merge_partial_and_user_variables(**kwargs)


In [22]:
print(len(summaries))

8


In [23]:
for summary in summaries:
    print(summary[0:100])
    print('-------------------')

The paper introduces **KULTURE Bench**, a benchmark designed to evaluate whether large language mode
-------------------
KULTURE Bench addresses the limitation of existing multilingual benchmarks that often rely on transl
-------------------
The paper introduces **KULTURE Bench**, a benchmark designed to evaluate how well language models un
-------------------
The paper introduces **KULTURE Bench**, a benchmark designed to evaluate how well language models un
-------------------
The paper introduces **KULTURE Bench**, a benchmark designed to evaluate how well language models un
-------------------
The paper introduces **KULTURE Bench**, a benchmark designed to evaluate whether language models can
-------------------
The paper introduces **KULTURE Bench**, a benchmark designed to evaluate whether language models can
-------------------
The paper introduces **KULTURE Bench**, a benchmark designed to evaluate how well language models un
-------------------


summaries의 내용을 하나로 결합한 뒤, 전체 요약을 생성하는 Reduce 체인을 구성하세요.   

최종 결과는 summary에 저장하세요.

In [24]:
# Reduce 과정 : 각 문서의 요약을 하나로 합칩니다.
reduce_prompt = ChatPromptTemplate([
    ('system', '''논문 요약문의 리스트가 주어집니다.
이를 읽고, 전체 주제를 포함하는 최종 요약을 한국어로 작성하세요.
요약은 8000자 이내로 작성하세요.

형식은 크게 2단 목차로 구성된 마크다운으로 작성하세요.

답변은 한국어로 작성하세요.'''),
    ('human', '''

논문 요약문의 리스트: {summaries}

''')
])

summaries_str = '\n\n---\n\n'.join(summaries)
# (요약1 엔터 엔터 --- 엔터 엔터 요약2 ---)


reduce_chain = reduce_prompt | llm | StrOutputParser()

summary = reduce_chain.invoke(summaries_str)
print(summary)

# 1. 연구 개요와 문제의식

## 1.1 연구 목적

이 논문은 대규모 언어모델이 단순히 한국어를 유창하게 처리하는지를 넘어서, **한국 문화적 맥락을 실제로 이해하고 추론할 수 있는지** 평가하기 위한 벤치마크인 **KULTURE Bench**를 제안한다. 기존 다국어 벤치마크는 영어 데이터셋을 번역해 만드는 경우가 많아, 서구 중심의 문화적 전제와 표현 방식이 그대로 반영될 위험이 있다. 따라서 이러한 벤치마크는 한국어 표면 능력이나 일반 지식은 평가할 수 있어도, 한국 문화에 특유한 관습, 표현, 역사적 배경, 문학적 감수성까지 측정하기에는 한계가 있다.

논문의 핵심 문제의식은 **언어 유창성 fluency과 문화 이해 cultural comprehension를 구분해야 한다**는 데 있다. 현대 LLM은 한국어로 자연스럽고 논리적으로 보이는 답변을 생성할 수 있지만, 그것이 반드시 한국 문화의 암묵적 의미, 관용적 표현, 문학적 뉘앙스, 사회적 맥락을 이해했다는 뜻은 아니다. KULTURE Bench는 이러한 숨은 약점을 드러내기 위해 한국 문화 자료를 직접 활용한 평가 체계를 구축한다.

## 1.2 기존 연구와의 차별성

기존에도 한국어 또는 한국 문화 관련 벤치마크가 존재한다. 예를 들어 KoBBQ는 편향 평가, HAE-RAE BENCH와 CLIcK는 문화 지식, KorNAT는 문화 정렬을 다룬다. 그러나 많은 기존 평가가 일반 질의응답이나 번역 기반 과제에 가까워, 문화적으로 내재된 언어 현상 자체를 충분히 반영하지 못한다는 한계가 있다.

KULTURE Bench는 한국어 텍스트 안에 실제로 녹아 있는 문화적 의미를 평가 대상으로 삼는다. 특히 다음 세 영역을 중심으로 한다.

- **한국어 사자성어 및 한자성어**
- **한국 시**
- **한국 문화 관련 뉴스**

이를 통해 모델이 단어 수준의 관용적 의미, 문장 수준의 문학적 흐름, 문단 수준의 사회·문화적 맥락을 얼마나 이해하는지 다층적으로 측정한다.

# 2. KULTURE Be

생성한 요약은 파일로 저장해도 좋습니다.

In [25]:
{result.metadata['Title']}

{'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context'}

In [26]:
with open(f'summary_{result.metadata['Title'][:4]}.md', 'w', encoding='utf-8') as f:
    f.write(summary)

<br><br><br><br><br><br>

### 성능 비교하기 (Future Work)

Stuff(전체를 입력하는 방식)과 비교하면 어떨까요?

In [27]:
prompt = ChatPromptTemplate([
    ('system', '''주어진 논문의 요약을 한국어로 작성하세요.
요약은 8000자 이내로 작성하세요.
답변은 한국어로 작성하세요.
'''),
    ('user', '''{text}
''')])

chain = prompt | llm | StrOutputParser()

original_summary = chain.invoke(result.page_content)

print(original_summary)

이 논문은 한국 문화 맥락에서 대형언어모델(LLM)의 문화 이해 및 추론 능력을 평가하기 위한 새로운 벤치마크인 **KULTURE Bench**를 제안한다. 기존 다국어 평가 벤치마크는 영어 데이터셋을 번역해 사용하는 경우가 많아 서구 문화 중심의 편향을 내포하며, 한국어와 한국 문화에 특화된 깊이 있는 평가에는 한계가 있다. 이에 저자들은 한국 문화가 반영된 뉴스, 사자성어, 시를 기반으로 총 **3,584개 문항**으로 구성된 평가 프레임워크를 구축하였다.

KULTURE Bench는 세 가지 하위 데이터셋으로 구성된다. 첫째, **KorID**는 한국어 사자성어 이해를 평가하는 데이터셋으로, 뉴스 기사 속 사자성어를 빈칸으로 가리고 다섯 개 후보 중 적절한 표현을 고르는 클로즈 테스트 형식이다. 총 1,631개 문항으로 구성되며, 사자성어의 문자적 의미가 아닌 역사적·문화적 배경과 비유적 의미를 이해해야 정답을 선택할 수 있다. 둘째, **KorPD**는 한국 교과서에 수록된 시를 바탕으로 한 데이터셋이다. 시의 한 행을 빈칸으로 만들고, 모델이 시 전체의 의미, 정서, 운율, 수사적 흐름을 고려해 올바른 행을 선택하도록 설계되었다. 총 453개 문항으로, 문장 수준의 문화적·문학적 이해 능력을 평가한다. 셋째, **KorCND**는 한국 문화 관련 뉴스 기사와 제목을 매칭하는 데이터셋이다. 모델은 기사 본문을 이해하고 요약한 뒤, 후보 제목 중 가장 적절한 헤드라인을 선택해야 한다. 총 1,500개 문항으로, 문단 수준의 문화적 이해와 요약 능력을 측정한다.

데이터 구축 과정에서는 실제 사용되는 한국어 자료의 진정성을 중시하였다. KorID는 사자성어 대사전 두 종에서 수집한 5,372개 사자성어를 기반으로 하며, 국립국어원 모두의 말뭉치 내 신문 코퍼스에서 실제 사용 사례를 추출하였다. 후보 오답은 ETRI의 KorBERT 임베딩을 이용해 의미적 유사도가 일정 범위에 속하는 표현을 선정함으로써 난이도를 조절하였다. KorPD는 한국 교과서의 시 91

Stuff와 Step-Back만 수행한 경우와도 비교해 보겠습니다.

In [28]:
prompt = ChatPromptTemplate([
    ('system', '''주어진 논문의 요약을 한국어로 작성하세요.
요약은 8000자 이내로 작성하세요.
답변은 한국어로 작성하세요.
'''),
    ('user', '''{text}
---

다음은 요약 가이드라인입니다.

{guide}''')]).partial(guide=guide)

chain = prompt | llm | StrOutputParser()

stepback_summary = chain.invoke(result.page_content)

print(stepback_summary)

/usr/local/lib/python3.12/dist-packages/langchain_core/prompts/chat.py:1187: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  kwargs = self._merge_partial_and_user_variables(**kwargs)


이 논문은 기존 다국어 언어모델 평가가 주로 영어 벤치마크를 번역해 사용하는 방식에 의존하며, 그 결과 서구 문화 중심의 편향을 포함할 수 있다는 문제의식에서 출발한다. 저자들은 단순한 한국어 처리 능력이 아니라, 한국 문화에 내재된 관습, 역사적 배경, 관용적 표현, 문학적 정서와 같은 심층적 문화 맥락을 대형언어모델(LLM)이 얼마나 이해하고 추론할 수 있는지를 평가하는 것이 필요하다고 주장한다. 이를 위해 한국 문화 맥락에 특화된 평가 프레임워크인 **KULTURE Bench**를 제안한다.

KULTURE Bench는 총 **3,584개 문항**으로 구성된 벤치마크로, 한국 문화 이해를 **단어, 문장, 문단 수준**에서 다층적으로 평가하도록 설계되었다. 세부적으로는 한국어 사자성어 이해를 평가하는 **KorID**, 한국 시의 문맥과 정서를 이해하는 능력을 평가하는 **KorPD**, 한국 문화 관련 뉴스 기사의 핵심 내용을 파악하고 적절한 제목을 고르는 **KorCND**로 구성된다. 이처럼 관용어, 시, 문화 뉴스라는 서로 다른 텍스트 유형을 포함함으로써, 모델이 표면적 언어 처리뿐 아니라 문화적 함의와 맥락적 의미를 얼마나 잘 이해하는지 평가할 수 있도록 했다.

첫 번째 데이터셋인 **KorID**는 1,631개 문항으로 이루어진 한국어 사자성어 빈칸 채우기 데이터셋이다. 사자성어는 문자 그대로의 의미만으로는 실제 의미를 파악하기 어려운 경우가 많으며, 역사적 일화나 문화적 배경을 알아야 정확히 이해할 수 있다. 예를 들어 “함흥차사”는 문자적으로는 “함흥에 보낸 사신”이지만, 실제로는 “심부름을 간 사람이 돌아오지 않거나 소식이 없는 상황”을 뜻한다. KorID는 사전에서 수집한 5,372개의 사자성어를 바탕으로 국립국어원 모두의 말뭉치 내 신문 기사에서 실제 사용 사례를 추출하고, KorBERT 임베딩 기반 코사인 유사도를 활용해 정답과 의미적으로 가까운 오답 후보를 구성했다. 이를 통해 모델이 문맥 속에서 사자성어의 확장 의미와 문화적 함

과거에는 요약 성능을 평가하기 위해 레퍼런스 요약문과 실제 요약을 키워드로 비교하는 ROUGE 등의 Score를 사용했으나,   

최근에는 LLM을 통해 평가하는 경우도 많습니다.

In [29]:
# 다양한 평가 기준을 세우고, Structured Output을 통해 평가하는 체인을 구성해도 좋습니다!

evaluation_prompt = ChatPromptTemplate(
    [
        ('system', '''논문의 원본 내용과, 이를 3가지 방법으로 요약한 요약문이 주어집니다.
세 요약문의 순위를 매기고, 그 이유를 설명하세요.

평가 기준은 다음과 같습니다.

- 논문에서 풀고자 하는 문제에 대해 정확히 파악했는가?
- 원문의 핵심 내용과 시사점을 잘 포함하고 있는가?
- 연구의 제약점과 향후 발전 방향에 대해 잘 기술하고 있는가?

'''),
        ('user', '''
원본 내용:
{original_text}


---

1번 요약문:
{summary1}

---

2번 요약문:
{summary2}

---

3번 요약문:
{summary3}
''')
])

evaluation_chain = evaluation_prompt | llm | StrOutputParser()

evaluation_result = evaluation_chain.invoke({
    'original_text': result.page_content,
    'summary1': summary,
    'summary2': stepback_summary,
    'summary3': original_summary
})

print(evaluation_result)

## 순위

**1위: 2번 요약문**  
**2위: 1번 요약문**  
**3위: 3번 요약문**

---

## 1위: 2번 요약문

2번 요약문이 가장 우수합니다. 원문의 문제의식, 벤치마크 구성, 실험 결과, CoT 분석, 인간 평가, 오류 분석, 한계와 향후 과제를 균형 있게 정리했습니다.

### 장점

- **논문이 풀고자 하는 문제를 정확히 파악함**
  - 기존 다국어 평가가 영어 벤치마크 번역에 의존하고, 그로 인해 서구 문화 편향이 발생할 수 있다는 핵심 문제를 잘 설명했습니다.
  - 단순 한국어 처리 능력이 아니라 한국 문화의 관습, 역사, 관용 표현, 문학적 정서까지 평가해야 한다는 논문의 목적을 정확히 짚었습니다.

- **핵심 내용과 실험 결과를 충실히 포함함**
  - KULTURE Bench의 전체 규모인 3,584개 문항과 세 하위 데이터셋 KorID, KorPD, KorCND의 목적과 구성 방식을 정확히 설명했습니다.
  - Clova X, GPT-4, GPT-3.5 Turbo, Tiangong의 비교 결과도 원문에 맞게 정리했습니다.
  - Clova X가 KorID와 KorCND에서 가장 우수하고, GPT-4가 KorPD에서 최고 성능을 보였다는 점을 정확히 반영했습니다.
  - 뉴스 과제는 상대적으로 쉽고, 사자성어와 시 과제는 어렵다는 핵심 시사점도 잘 담았습니다.

- **CoT, 인간 평가, 오류 분석까지 균형 있게 다룸**
  - CoT가 문화 지식이 충분한 모델에서는 도움이 되지만, 지식이 부족한 모델에서는 오히려 잘못된 추론을 강화할 수 있다는 논문의 중요한 분석을 잘 요약했습니다.
  - 인간 평가에서 전문가와 비전문가 모두 LLM보다 전반적으로 높은 성능을 보였다는 점도 포함했습니다.
  - 의미 이해 오류, 문화 배경지식 부족, 문법·어휘 오류, 논리 오류, 문맥 해석 부족이라는 오류 유형도 정확히 정리했습니다.

- **한계와 향후 연구 방향도 적절함**
  - 평가 모델 수 제한, CoT 실험의 범위